In [11]:
import cv2
import os
import re
import numpy as np
import math
from IPython.display import Video, display
from google.colab.patches import cv2_imshow
import base64
from IPython.display import HTML, display

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nikanvasei/shanghaitech-campus-dataset")

print("Path to dataset files:", path)

Resuming download from 207618048 bytes (11510876230 bytes left)...
Resuming download to /root/.cache/kagglehub/datasets/nikanvasei/shanghaitech-campus-dataset/1.archive (207618048/11718494278) bytes left.


100%|██████████| 10.9G/10.9G [04:46<00:00, 40.2MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/nikanvasei/shanghaitech-campus-dataset/versions/1


In [7]:
import os
import cv2

class VideoSequence:

	def __init__(self, video_path):

		self.video_path = video_path

		self.frame_paths = sorted(
			[
				os.path.join(video_path, file)
				for file in os.listdir(video_path)
				if file.endswith(".jpg")
			]
		)

	def __len__(self):
		return len(self.frame_paths)

	def get_frame(self, index):
		return cv2.imread(self.frame_paths[index])

	def get_frame_path(self, index):
		return self.frame_paths[index]

class VideoDataset:

	def __init__(self, root):

		self.videos = []

		for directory in sorted(os.listdir(root)):

			video_path = os.path.join(root, directory)

			if os.path.isdir(video_path):

				self.videos.append(
					VideoSequence(video_path)
				)

	def __len__(self):
		return len(self.videos)

	def get_video(self, index):
		return self.videos[index]

In [ ]:
def video_to_mp4(video, output_file="output.webm", fps=30):
    first_frame = video.get_frame(0) if hasattr(video, 'get_frame') else video[0]
    height, width = first_frame.shape[:2]
    is_color = len(first_frame.shape) == 3

    if not output_file.endswith('.webm'):
        output_file = output_file.rsplit('.', 1)[0] + '.webm'

    fourcc = cv2.VideoWriter_fourcc(*"VP80")
    # ------------------------------------

    writer = cv2.VideoWriter(
        output_file,
        fourcc,
        fps,
        (width, height),
        isColor=is_color
    )

    for i in range(len(video)):
        frame = video.get_frame(i) if hasattr(video, 'get_frame') else video[i]
        writer.write(frame)

    writer.release()
    print(f"Vídeo salvo com sucesso em: {output_file}")

    with open(output_file, 'rb') as f:
        video_bytes = f.read()
    video_url = f"data:video/webm;base64,{base64.b64encode(video_bytes).decode()}"

    html_player = f"""
    <video width="{width}" height="{height}" controls autoplay loop>
          <source src="{video_url}" type="video/webm">
          Seu navegador não suporta a exibição deste vídeo.
    </video>
    """

    display(HTML(html_player))

    return output_file

def convert_video_to_grayscale(video):
  gray_frames = []
  for i in range(len(video)):
    frame = video.get_frame(i)
    gray = cv2.cvtColor(
		  frame,
		  cv2.COLOR_BGR2GRAY
	  )
    gray_frames.append(gray)
  return gray_frames